# Binary Classifier — CBB vs Healthy
**Task:** Binary classification of Cassava Bacterial Blight (CBB) vs Healthy leaves  
**Backbone options:** EfficientNetB0 (NoisyStudent) or ConvNeXtBase  
**Protocol:** 5-fold stratified CV, CLAHE preprocessing, 3-phase training, CBB oversampling

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import os
import cv2
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print(f'TensorFlow: {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ── Cell 2: CONFIG — update paths before running ─────────────────────────────
CONFIG = {
    # Choose backbone: 'efficientnet' or 'convnext'
    'backbone':               'efficientnet',

    # ── Paths — UPDATE THESE ──────────────────────────────────────────────────
    'image_dir':              r'C:\path\to\cassava\train_images',
    'train_csv':              r'C:\path\to\cassava\train.csv',
    'clahe_dir':              r'C:\path\to\cassava\clahe_binary',
    'noisy_student_weights':  r'C:\path\to\efficientnet-b0_noisy-student_notop.h5',
    'save_dir':               r'C:\path\to\save\models',

    # ── Training ──────────────────────────────────────────────────────────────
    'img_size':               384,
    'batch_size':             32,
    'num_classes':            2,
    'num_folds':              5,
    'seed':                   42,
    'cbb_oversample':         3,    # multiply CBB samples by this factor

    # Phase 1 — frozen backbone
    'epochs_phase1':          25,
    'lr_phase1':              1e-4,
    'patience_phase1':        7,

    # Phase 2A — partial unfreeze (top 30 layers)
    'epochs_phase2a':         20,
    'lr_phase2a':             1e-5,
    'patience_phase2a':       7,

    # Phase 2B — full unfreeze
    'epochs_phase2b':         10,
    'lr_phase2b':             5e-6,
    'patience_phase2b':       6,
}

tf.random.set_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
os.makedirs(CONFIG['save_dir'], exist_ok=True)
os.makedirs(CONFIG['clahe_dir'], exist_ok=True)
print('CONFIG loaded')
print(f'Backbone: {CONFIG["backbone"]}')
print(f'Image size: {CONFIG["img_size"]}x{CONFIG["img_size"]}')

In [ ]:
# ── Cell 3: Mixed precision ───────────────────────────────────────────────────
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
print('Mixed precision: mixed_float16 enabled')

In [ ]:
# ── Cell 4: Load dataset and filter CBB + Healthy ────────────────────────────
df = pd.read_csv(CONFIG['train_csv'])
df['image_path'] = df['image_id'].apply(
    lambda x: os.path.join(CONFIG['image_dir'], x))

# Original labels: CBB=0, Healthy=4
# Remapped:        CBB=0, Healthy=1
df_binary = df[df['label'].isin([0, 4])].copy().reset_index(drop=True)
df_binary['label'] = df_binary['label'].map({0: 0, 4: 1})

print(f'Total binary samples: {len(df_binary)}')
print(f'  CBB (0):     {(df_binary["label"]==0).sum()}')
print(f'  Healthy (1): {(df_binary["label"]==1).sum()}')
print(f'  Imbalance ratio: {(df_binary["label"]==1).sum() / (df_binary["label"]==0).sum():.2f}x')

In [ ]:
# ── Cell 5: CLAHE preprocessing ──────────────────────────────────────────────
def apply_clahe(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img[:, :, 0] = clahe.apply(img[:, :, 0])
    img = cv2.cvtColor(img, cv2.COLOR_LAB2RGB)
    return img

print('Applying CLAHE...')
skipped = 0
for idx, row in df_binary.iterrows():
    out_path = os.path.join(CONFIG['clahe_dir'], row['image_id'])
    if not os.path.exists(out_path):
        img = apply_clahe(row['image_path'])
        if img is not None:
            cv2.imwrite(out_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
        else:
            skipped += 1
    if idx % 200 == 0:
        print(f'  {idx}/{len(df_binary)}')

df_binary['clahe_path'] = df_binary['image_id'].apply(
    lambda x: os.path.join(CONFIG['clahe_dir'], x))

# Verify paths exist
missing = df_binary['clahe_path'].apply(lambda x: not os.path.exists(x)).sum()
print(f'CLAHE done — skipped {skipped} corrupt | missing paths: {missing}')

In [ ]:
# ── Cell 6: Class weights ─────────────────────────────────────────────────────
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(df_binary['label']),
    y=df_binary['label']
)
class_weight_dict = dict(enumerate(class_weights))
print('Class weights:', class_weight_dict)

In [ ]:
# ── Cell 7: Preprocessing and dataset functions ───────────────────────────────
def preprocess_efficientnet(image_path, label):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [CONFIG['img_size'], CONFIG['img_size']])
    img.set_shape([CONFIG['img_size'], CONFIG['img_size'], 3])
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img, label

def preprocess_convnext(image_path, label):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [CONFIG['img_size'], CONFIG['img_size']])
    img.set_shape([CONFIG['img_size'], CONFIG['img_size'], 3])
    img = tf.keras.applications.convnext.preprocess_input(img)
    return img, label

def augment_image_tf(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.2)
    img = tf.image.random_contrast(img, lower=0.8, upper=1.2)
    img = tf.image.random_saturation(img, lower=0.8, upper=1.2)
    crop_size = tf.cast(
        tf.cast(CONFIG['img_size'], tf.float32) *
        tf.random.uniform([], 0.85, 1.0), tf.int32)
    img = tf.image.random_crop(img, size=[crop_size, crop_size, 3])
    img = tf.image.resize(img, [CONFIG['img_size'], CONFIG['img_size']])
    return img, label

def build_dataset(image_paths, labels, augment=False):
    preprocess_fn = (preprocess_efficientnet
                     if CONFIG['backbone'] == 'efficientnet'
                     else preprocess_convnext)
    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    ds = ds.map(preprocess_fn, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_image_tf, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.shuffle(1024)
    ds = ds.batch(CONFIG['batch_size']).prefetch(tf.data.AUTOTUNE)
    return ds

def build_dataset_eval(image_paths, labels):
    preprocess_fn = (preprocess_efficientnet
                     if CONFIG['backbone'] == 'efficientnet'
                     else preprocess_convnext)
    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    ds = ds.map(preprocess_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(CONFIG['batch_size']).prefetch(tf.data.AUTOTUNE)
    return ds

print('Dataset functions ready')

In [ ]:
# ── Cell 8: Model builder ─────────────────────────────────────────────────────
def build_model(trainable_base=False):
    if CONFIG['backbone'] == 'efficientnet':
        base = tf.keras.applications.EfficientNetB0(
            input_shape=(CONFIG['img_size'], CONFIG['img_size'], 3),
            include_top=False,
            weights=None
        )
        base.load_weights(
            CONFIG['noisy_student_weights'],
            by_name=True,
            skip_mismatch=True
        )
        print('  NoisyStudent weights loaded')
        norm_layer = tf.keras.layers.BatchNormalization()
    else:
        base = tf.keras.applications.ConvNeXtBase(
            input_shape=(CONFIG['img_size'], CONFIG['img_size'], 3),
            include_top=False,
            weights='imagenet'
        )
        print('  ConvNeXtBase ImageNet weights loaded')
        norm_layer = tf.keras.layers.LayerNormalization()

    base.trainable = trainable_base
    inputs  = tf.keras.Input(shape=(CONFIG['img_size'], CONFIG['img_size'], 3))
    x       = base(inputs, training=False)
    x       = tf.keras.layers.GlobalAveragePooling2D()(x)
    x       = norm_layer(x)
    x       = tf.keras.layers.Dropout(0.3)(x)
    x       = tf.keras.layers.Dense(128, activation='relu')(x)
    x       = tf.keras.layers.Dropout(0.2)(x)
    x       = tf.keras.layers.Dense(CONFIG['num_classes'])(x)
    outputs = tf.keras.layers.Activation('softmax', dtype='float32')(x)
    model   = tf.keras.Model(inputs, outputs)
    print(f'  Total params: {model.count_params():,}')
    return model

print('Model builder ready')

In [ ]:
# ── Cell 9: CBB oversampling ──────────────────────────────────────────────────
def oversample_cbb(train_df, multiplier=3):
    cbb_df = train_df[train_df['label'] == 0]
    oversampled = pd.concat(
        [train_df] + [cbb_df] * (multiplier - 1)
    ).sample(frac=1, random_state=CONFIG['seed']).reset_index(drop=True)
    print(f'  After oversample — CBB: {(oversampled["label"]==0).sum()} | '
          f'Healthy: {(oversampled["label"]==1).sum()}')
    return oversampled

print('Oversample function ready')

In [ ]:
# ── Cell 10: Training loop ────────────────────────────────────────────────────
skf = StratifiedKFold(
    n_splits=CONFIG['num_folds'],
    shuffle=True,
    random_state=CONFIG['seed']
)

fold_accuracies = []
all_preds       = []
all_labels      = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(df_binary['clahe_path'], df_binary['label'])
):
    print(f'\n{"="*55}')
    print(f'FOLD {fold+1}/{CONFIG["num_folds"]}  |  Backbone: {CONFIG["backbone"]}')
    print(f'{"="*55}')

    train_df = df_binary.iloc[train_idx].copy()
    val_df   = df_binary.iloc[val_idx].copy()

    # Oversample CBB in training only
    train_df    = oversample_cbb(train_df, CONFIG['cbb_oversample'])
    train_ds    = build_dataset(train_df['clahe_path'].values,
                                train_df['label'].values, augment=True)
    val_ds      = build_dataset(val_df['clahe_path'].values,
                                val_df['label'].values, augment=False)
    val_ds_eval = build_dataset_eval(val_df['clahe_path'].values,
                                     val_df['label'].values)

    # ── Phase 1: frozen backbone ──────────────────────────────────────────────
    print('\n[Phase 1] Training head — backbone frozen')
    model = build_model(trainable_base=False)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(CONFIG['lr_phase1']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    model.fit(
        train_ds, validation_data=val_ds,
        epochs=CONFIG['epochs_phase1'],
        class_weight=class_weight_dict,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor='val_accuracy', mode='max',
                patience=CONFIG['patience_phase1'],
                restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.2,
                patience=4, min_lr=1e-7, verbose=1)
        ], verbose=1
    )

    # ── Phase 2A: partial unfreeze ────────────────────────────────────────────
    print('\n[Phase 2A] Partial unfreeze — top 30 backbone layers')
    base_model = model.layers[1]
    for layer in base_model.layers[-30:]:
        layer.trainable = True

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            CONFIG['lr_phase2a'], clipnorm=1.0),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    model.fit(
        train_ds, validation_data=val_ds,
        epochs=CONFIG['epochs_phase2a'],
        class_weight=class_weight_dict,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor='val_accuracy', mode='max',
                patience=CONFIG['patience_phase2a'],
                restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.2,
                patience=4, min_lr=1e-7, verbose=1)
        ], verbose=1
    )

    # ── Phase 2B: full unfreeze ───────────────────────────────────────────────
    print('\n[Phase 2B] Full unfreeze')
    for layer in base_model.layers:
        layer.trainable = True

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            CONFIG['lr_phase2b'], clipnorm=1.0),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    model.fit(
        train_ds, validation_data=val_ds,
        epochs=CONFIG['epochs_phase2b'],
        class_weight=class_weight_dict,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor='val_accuracy', mode='max',
                patience=CONFIG['patience_phase2b'],
                restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.2,
                patience=3, min_lr=1e-7, verbose=1)
        ], verbose=1
    )

    # ── Save ──────────────────────────────────────────────────────────────────
    save_path = os.path.join(
        CONFIG['save_dir'],
        f'fold_{fold+1}_binary_cbb_healthy_{CONFIG["backbone"]}.keras'
    )
    model.save(save_path)
    print(f'Saved: {save_path}')

    # ── Evaluate ──────────────────────────────────────────────────────────────
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    probs  = model.predict(val_ds_eval, verbose=0)
    preds  = np.argmax(probs, axis=1)
    labels = val_df['label'].values

    acc = np.mean(preds == labels)
    fold_accuracies.append(acc)
    all_preds.extend(preds.tolist())
    all_labels.extend(labels.tolist())

    print(f'\nFold {fold+1} Accuracy: {acc:.4f}')
    print(classification_report(labels, preds,
          target_names=['CBB', 'Healthy']))

    del model
    tf.keras.backend.clear_session()
    gc.collect()

In [ ]:
# ── Cell 11: Final summary ────────────────────────────────────────────────────
print(f'\n{"="*55}')
print('FINAL RESULTS — Binary CBB vs Healthy')
print(f'Backbone: {CONFIG["backbone"]}')
print(f'{"="*55}')
print(f'Per-fold accuracies: {[f"{a:.4f}" for a in fold_accuracies]}')
print(f'Mean Accuracy: {np.mean(fold_accuracies):.4f} '
      f'(+/- {np.std(fold_accuracies):.4f})')
print(f'\nPer-class metrics (all folds combined):')
print(classification_report(all_labels, all_preds,
      target_names=['CBB', 'Healthy']))

In [ ]:
# ── Cell 12: Confusion matrix ─────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['CBB', 'Healthy'],
            yticklabels=['CBB', 'Healthy'],
            ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix (Counts)')

# Percentages
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=['CBB', 'Healthy'],
            yticklabels=['CBB', 'Healthy'],
            ax=axes[1])
for text in axes[1].texts:
    text.set_text(text.get_text() + '%')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix (% per class)')

plt.suptitle(
    f'Binary Classifier — CBB vs Healthy\n'
    f'Backbone: {CONFIG["backbone"]}  |  '
    f'Mean Acc: {np.mean(fold_accuracies):.4f} '
    f'(±{np.std(fold_accuracies):.4f})',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
save_cm = os.path.join(CONFIG['save_dir'],
    f'binary_confusion_{CONFIG["backbone"]}.png')
plt.savefig(save_cm, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {save_cm}')